# Exploration de l'API TED (marchés publics européens)

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)
**Module :** 1 - Collecte
**Source :** TED (Tenders Electronic Daily), le journal officiel des marchés publics de l'UE.

## Pourquoi TED

La BCEAO publie très peu de cybersécurité. Je cherche une source avec du volume et de vraies
opportunités cyber. TED est une **API officielle** : données structurées (pas de HTML à scraper),
énorme volume (toute l'Europe), filtrage par mots-clés / pays / catégorie.

## Ma démarche

Je ne connais pas cette API. Je la découvre **par tâtonnements** : je tente un appel, je lis ce que
l'API répond (surtout ses erreurs, très instructives), et j'ajuste. Je veux qu'aucun nom de champ
n'apparaisse "par magie" : chaque nom que j'utilise, je montre d'abord d'où il vient.

## 1. Est-ce que l'API répond ?

Je fais l'appel le plus simple possible et je regarde le **code HTTP** (200 = OK, 4xx = j'ai fait
une erreur, 5xx = souci serveur).

In [24]:
import requests
import json
from collections import Counter
from datetime import date

URL = "https://api.ted.europa.eu/v3/notices/search"

r = requests.post(URL, json={"query": "publication-date>=today(-1)"})
print("Code HTTP :", r.status_code)

Code HTTP : 400


**400** : l'API n'est pas contente. Ce n'est pas un échec, c'est une piste : elle va me dire
ce qui manque. Je lis le message.

In [25]:
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

{
  "message": "Validation error",
  "error": [
    {
      "objectName": "publicExpertSearchRequestV1",
      "field": "fields",
      "message": "must not be empty"
    }
  ]
}


Le message dit : le champ **`fields` ne doit pas être vide**. L'API veut que je précise quelles
informations je veux pour chaque annonce. Mais je ne connais aucun nom de champ valide... Je vais
en tenter un au hasard pour voir comment l'API réagit.

## 2. Découvrir les noms de champs (grâce à l'erreur)

Je tente un nom intuitif au hasard : `"title"`.

In [26]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)", "fields": ["title"]})
print("Code HTTP :", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False)[:350])

Code HTTP : 400
{"message": "Parameter 'fields' contains unsupported value (supported values are: sme-part,touchpoint-gateway-ted-esen,submission-url-lot,organisation-person-addinfo-part,no-negocaition-necessary-lot,BT-13(t)-Part,organisation-city-serv-prov,result-framework-maximum-value-cur-notice,BT-821-Lot,touchpoint-partname-tenderer,touchpoint-internet-addres


Encore 400, mais le message est en or : « unsupported value (**supported values are:** ... ) »
suivi d'une longue liste. En me trompant, l'API m'a **listé tous les champs valides**. Je récupère
cette liste et je la compte.

In [27]:
message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = [c.strip() for c in liste_brute.split(",") if c.strip()]
print("Nombre de champs proposés par l'API :", len(champs))
print("Exemples (10 premiers) :", champs[:5], "...")

Nombre de champs proposés par l'API : 1830
Exemples (10 premiers) : ['sme-part', 'touchpoint-gateway-ted-esen', 'submission-url-lot', 'organisation-person-addinfo-part', 'no-negocaition-necessary-lot'] ...


**Attention :** ces 1830, ce sont des **champs** (les colonnes possibles : titre, date, pays...),
PAS le nombre d'annonces. Je verrai le nombre d'annonces plus loin. Pour l'instant, je cherche dans
cette liste les champs dont j'ai besoin.

## 3. Chercher mes champs dans la liste (avant de les utiliser)

Je ne veux pas utiliser un nom de champ sans l'avoir vu dans la liste. Donc je fouille la liste avec
des mots-clés correspondant aux informations utiles : identifiant, titre, date, pays, lien, catégorie.
J'écarte les champs préfixés `BT-` (codes techniques peu lisibles) juste pour la lisibilité.

In [28]:
for mot in ["publication", "number", "title", "deadline", "buyer-country", "buyer-name", "links", "cpv"]:
    trouves = [c for c in champs if mot in c.lower() and not c.startswith("BT-")][:5]
    print(f"{mot:16} -> {trouves}")

publication      -> ['publication-date', 'publication-number', 'publication-name', 'notice-preferred-publication-date']
number           -> ['award-criterion-number-fixed-glo', 'framework-maximum-participants-number-lot', 'number-tender-applications-ipi-measure-res', 'OPA-36-Part-Number', 'award-criterion-number-fixed-lot']
title            -> ['review-title', 'announcement-title', 'contract-title', 'notice-title', 'title-part']
deadline         -> ['deadline-receipt-tender-date-lot', 'tender-validity-deadline-unit-lot', 'deadline-time-part', 'deadline-receipt-expressions-time-lot', 'deadline-receipt-request']
buyer-country    -> ['buyer-country-sub', 'buyer-country']
buyer-name       -> ['buyer-name']
links            -> ['links']
cpv              -> ['classification-cpv']


Maintenant je vois d'où viennent les noms que je vais utiliser. En particulier,
**`publication-number`** apparaît bien dans la liste (sous "publication" et "number") : c'est
l'identifiant de l'annonce. Je retiens : `publication-number`, `notice-title`, `buyer-name`,
`buyer-country`, `publication-date`, `deadline`, `classification-cpv`, `links`.

## 4. Premier appel qui marche, et surtout : que renvoie l'API ?

J'utilise `publication-number` (trouvé dans la liste) pour faire un appel valide. Puis, au lieu de
supposer la forme de la réponse, je **regarde les clés** que l'API me renvoie.

In [29]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)",
                              "fields": ["publication-number"], "limit": 3})
reponse = r.json()
print("Code HTTP :", r.status_code)
print("\nLes CLÉS que l'API me renvoie :")
for cle, val in reponse.items():
    apercu = f"liste de {len(val)} éléments" if isinstance(val, list) else val
    print(f"   {cle} = {apercu}")

Code HTTP : 200

Les CLÉS que l'API me renvoie :
   notices = liste de 3 éléments
   totalNoticeCount = 7321
   iterationNextToken = None
   timedOut = False


Voilà la structure de la réponse, sans rien deviner :
- **`notices`** : la liste des annonces demandées (ici 3, à cause de `limit=3`) ;
- **`totalNoticeCount`** : le nombre **total** d'annonces correspondant à ma recherche ;
- **`iterationNextToken`** : un jeton pour récupérer la suite (pagination) ;
- **`timedOut`** : si la requête a expiré.

C'est ici que je découvre **`totalNoticeCount`** : ce n'est pas un champ que je demande, c'est une
clé que l'API met dans sa réponse. C'est lui qui me donne le nombre de **lignes** (annonces).

## 5. Combien d'annonces existent ? (lignes, pas champs)

Je me fais une petite fonction pour lire `totalNoticeCount` selon une recherche.

In [30]:
def total_annonces(query, scope="ACTIVE"):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": scope}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Annonces publiées depuis 1 jour  :", total_annonces("publication-date>=today(-1)"))
print("Annonces publiées depuis 7 jours :", total_annonces("publication-date>=today(-7)"))

Annonces publiées depuis 1 jour  : 7320
Annonces publiées depuis 7 jours : 21952


Plusieurs milliers par jour. Sur le **site** TED, sans filtre, j'obtiens ~6 millions. Pourquoi
l'écart ? Parce que le site montre **tout l'historique**, alors que je filtre sur les derniers jours.
Je le vérifie en enlevant le filtre de date (scope ALL).

In [31]:
def total_all(query):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": "ALL"}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Tout l'historique :", total_all("publication-date>=today(-9000)"))
print("Depuis 1 an       :", total_all("publication-date>=today(-365)"))
print("Depuis 1 jour     :", total_all("publication-date>=today(-1)"))

Tout l'historique : 6937929
Depuis 1 an       : 900320
Depuis 1 jour     : 7321


Confirmé : tout l'historique donne ~6-7 millions, comme le site. L'API et le site sont donc
cohérents. **Conséquence :** pour ma veille, je filtrerai toujours par date récente + cyber + scope
ACTIVE, et je ferai tourner la collecte chaque jour (sinon le volume est ingérable).

## 6. Une fonction de recherche réutilisable

In [32]:
CHAMPS = ["publication-number", "notice-title", "buyer-name", "buyer-country",
          "publication-date", "deadline", "notice-type", "classification-cpv", "links"]

def chercher(query, limit=50, scope="ACTIVE"):
    body = {"query": query, "fields": CHAMPS, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=40)
    r.raise_for_status()
    return r.json()

## 7. Regarder une annonce en détail

Avant de filtrer, je regarde la structure brute d'une annonce (c'est là qu'on trouve les surprises).

In [33]:
data = chercher("publication-date>=today(-2)", limit=5)
print(json.dumps(data["notices"][0], indent=2, ensure_ascii=False)[:1000])

{
  "notice-type": "cn-standard",
  "publication-number": "413513-2026",
  "classification-cpv": [
    "60112000",
    "60112000"
  ],
  "buyer-name": {
    "deu": [
      "Zweckverband Verkehrsverbund Bremen/Niedersachsen (ZVBN)"
    ]
  },
  "publication-date": "2026-06-17+02:00",
  "links": {
    "xml": {
      "MUL": "https://ted.europa.eu/en/notice/413513-2026/xml"
    },
    "pdf": {
      "BUL": "https://ted.europa.eu/bg/notice/413513-2026/pdf",
      "SPA": "https://ted.europa.eu/es/notice/413513-2026/pdf",
      "CES": "https://ted.europa.eu/cs/notice/413513-2026/pdf",
      "DAN": "https://ted.europa.eu/da/notice/413513-2026/pdf",
      "DEU": "https://ted.europa.eu/de/notice/413513-2026/pdf",
      "EST": "https://ted.europa.eu/et/notice/413513-2026/pdf",
      "ELL": "https://ted.europa.eu/el/notice/413513-2026/pdf",
      "ENG": "https://ted.europa.eu/en/notice/413513-2026/pdf",
      "FRA": "https://ted.europa.eu/fr/notice/413513-2026/pdf",
      "GLE": "https://ted.europ

J'observe trois formats à gérer : le **titre** est un dictionnaire par langue, les **dates** ont
un fuseau (ex. `2026-06-18+02:00`), et `classification-cpv` est une **liste** de codes.

## 8. Isoler la cybersécurité : explorer la hiérarchie des CPV

Sur le site, à gauche, il y a des filtres par grandes catégories (« Computer and Related Services »,
etc.) avec un nombre d'annonces. Je me demande si l'API expose ces catégories. En fait non : l'API
raisonne en **codes CPV**. Mais je peux vérifier que les catégories du site correspondent à des
**préfixes CPV**, en comparant les nombres.

In [34]:
# def total_all(query):
#     body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": "ALL"}
#     r = requests.post(URL, json=body)
#     print("  statut HTTP:", r.status_code)        # pour diagnostiquer
#     print("  début réponse:", r.text[:200])       # voir ce qui est vraiment renvoyé
#     return r.json().get("totalNoticeCount")

In [35]:
# Le site indique : IT services = 390380, Software = 245508, Office machinery = 254720
print("CPV 72* (IT services)        :", total_all("classification-cpv=72*"), " (site: 390380)")
print("CPV 48* (Software & systems) :", total_all("classification-cpv=48*"), " (site: 245508)")
print("CPV 30* (Office machinery)   :", total_all("classification-cpv=30*"), " (site: 254720)")

CPV 72* (IT services)        : 390380  (site: 390380)
CPV 48* (Software & systems) : 245508  (site: 245508)
CPV 30* (Office machinery)   : 254720  (site: 254720)


Correspondance parfaite : les catégories lisibles du site sont des **préfixes CPV**. Le CPV est
hiérarchique (plus le code est long, plus c'est précis). Je descends dans « IT services » (72*) pour
trouver ce qui concerne l'audit et les tests, le cœur de métier de PWN & PATCH.

In [36]:
# Sur le site, en descendant : "Computer audit and testing services" = 2812
print("CPV 728*  (audit & testing)      :", total_all("classification-cpv=728*"), " (site: 2812)")
print("CPV 72810000 (computer audit)    :", total_all("classification-cpv=72810000"), " (site: 1010)")
print("CPV 72820000 (computer testing)  :", total_all("classification-cpv=72820000"), " (site: 304)")

CPV 728*  (audit & testing)      : 2812  (site: 2812)
CPV 72810000 (computer audit)    : 1010  (site: 1010)
CPV 72820000 (computer testing)  : 304  (site: 304)


Tout correspond à l'unité près. J'ai donc décodé la hiérarchie :

```
Computer and Related Services
└── IT services .......................... 72*        (390380)
    └── Computer audit and testing ....... 728*       (2812)
        ├── Computer audit services ...... 72810000   (1010)
        └── Computer testing services .... 72820000   (304)
```

Les codes **72810000** (audit) et **72820000** (testing) sont très pertinents pour PWN & PATCH.
**Mais** ils sont étroits : beaucoup d'annonces cyber sont classées ailleurs (sécurité réseau,
conformité, logiciels de sécurité en 48*, services de sécurité en 79*). Je ne peux donc pas me
limiter à ces codes : je vais les **combiner avec une recherche par mots-clés**.

## 9. Ma requête de collecte cyber (CPV ciblés + mots-clés)

Je combine les codes CPV d'audit/test (signal fort) ET des mots-clés dans le titre, en me limitant
aux mises en concurrence (`notice-type=cn-standard`), c'est-à-dire des appels d'offres ouverts (pas
des avis d'attribution déjà clôturés).

In [38]:
REQUETE_CYBER = (
    '(classification-cpv=72810000 OR classification-cpv=72820000 '
    'OR notice-title ~ "cybersecurity" OR notice-title ~ "penetration" '
    'OR notice-title ~ "ISO 27001" OR notice-title ~ "security audit" '
    'OR notice-title ~ "vulnerability" OR notice-title ~ "pentest") '
    'AND notice-type=cn-standard SORT BY publication-date DESC'
)
data = chercher(REQUETE_CYBER, limit=100)
print("Annonces cyber (mises en concurrence) :", data["totalNoticeCount"])
print("Récupérées dans l'échantillon :", len(data["notices"]))

Annonces cyber (mises en concurrence) : 191
Récupérées dans l'échantillon : 100


## 10. Contrôle qualité des données

Avant de coder le collecteur, je vérifie la qualité sur cet échantillon réel.

In [39]:
notices = data["notices"]
n = len(notices)
print(f"Échantillon : {n} annonces\n")
print("a) Complétude :")
for champ in CHAMPS:
    presents = sum(1 for x in notices if x.get(champ) not in (None, "", [], {}))
    print(f"   {champ:20} : {presents}/{n}  ({100*presents//max(n,1)}%)")

Échantillon : 100 annonces

a) Complétude :
   publication-number   : 100/100  (100%)
   notice-title         : 100/100  (100%)
   buyer-name           : 100/100  (100%)
   buyer-country        : 100/100  (100%)
   publication-date     : 100/100  (100%)
   deadline             : 12/100  (12%)
   notice-type          : 100/100  (100%)
   classification-cpv   : 100/100  (100%)
   links                : 100/100  (100%)


In [40]:
print("b) Doublons sur publication-number :")
nums = [x.get("publication-number") for x in notices]
print(f"   {len(nums)} numéros, {len(set(nums))} uniques -> {len(nums)-len(set(nums))} doublon(s)")

b) Doublons sur publication-number :
   100 numéros, 100 uniques -> 0 doublon(s)


In [41]:
print("c) Langues des titres :")
sans_anglais = sum(1 for x in notices
                   if isinstance(x.get("notice-title"), dict) and "eng" not in x["notice-title"])
print(f"   annonces sans titre anglais : {sans_anglais}/{n}")

c) Langues des titres :
   annonces sans titre anglais : 0/100


In [42]:
print("d) Date limite (deadline) :")
def premiere(d): return d[0] if isinstance(d, list) else d
avec_dl = [x for x in notices if x.get("deadline")]
print(f"   présente dans : {len(avec_dl)}/{n}")
if avec_dl:
    print("   exemple de format :", premiere(avec_dl[0]["deadline"]))
    auj = date.today().isoformat()
    passees = sum(1 for x in avec_dl if str(premiere(x["deadline"]))[:10] < auj)
    print(f"   déjà passées alors que scope=ACTIVE : {passees}/{len(avec_dl)}")

d) Date limite (deadline) :
   présente dans : 12/100
   exemple de format : 2026-06-29T15:00:00+01:00
   déjà passées alors que scope=ACTIVE : 9/12


In [43]:
print("e) Lien : je le reconstruis depuis le numéro")
num = notices[0]["publication-number"]
print("   ", f"https://ted.europa.eu/fr/notice/{num}")

e) Lien : je le reconstruis depuis le numéro
    https://ted.europa.eu/fr/notice/424465-2026


### Ce que j'apprends

- Champs **fiables à 100 %** : `publication-number`, `notice-title`, `buyer-name`,
  `buyer-country`, `publication-date`, `classification-cpv`, `links`.
- **`publication-number` unique** (0 doublon) -> ma clé.
- **Titres toujours en anglais** -> je prends `eng`.
- **`deadline` souvent absente, et parfois déjà passée même en scope ACTIVE.** Je ne m'y fie donc
  pas seule : si elle est présente et dépassée, l'annonce est close ; sinon je me base sur `ACTIVE`.
  Format à normaliser : `2026-06-29T15:00:00+01:00` -> `2026-06-29`.
- **Liens** : structure complexe -> je reconstruis `https://ted.europa.eu/fr/notice/<num>`.

## 11. Les détails riches (prix, procédure, gagnant) sont-ils dans l'API ?

En cliquant sur une annonce du site, j'ai vu plein de détails : valeur estimée (le prix), type de
procédure, durée, gagnant, email de l'acheteur. Est-ce que l'API a ces champs ? Je cherche dans la
liste.

In [44]:
for label, ms in {"valeur/prix":["estimated-value","value-cur"], "procédure":["procedure-type"],
                  "durée":["contract-duration"], "gagnant":["winner"],
                  "description":["description-lot","description-proc"]}.items():
    trouves = []
    for m in ms:
        trouves += [c for c in champs if m in c.lower() and not c.startswith("BT-")]
    print(f"{label:14} -> {trouves[:4]}")

valeur/prix    -> ['framework-estimated-value-glo', 'group-framework-re-estimated-value-cur-res', 'estimated-value-cur-lot', 'framework-estimated-value-cur-glo']
procédure      -> ['procedure-type']
durée          -> ['contract-duration-period-oth-part', 'contract-duration-start-date-part', 'contract-duration-end-date-part', 'contract-duration-period-oth-lot']
gagnant        -> ['winner-post-code', 'winner-country', 'winner-touchpoint-post-code', 'winner-gateway']
description    -> ['option-description-lot', 'missing-info-submission-description-lot', 'selection-criterion-description-lot', 'renewal-description-lot']


Oui, l'API contient ces détails. Donc **pas besoin de scraper la page de détail** : tout est
dans l'API, il suffit d'ajouter les champs voulus à ma requête.

**Stratégie (la même que pour la BCEAO) :** je ne récupère pas tout pour tout le monde.
- À la collecte : champs **essentiels** (titre, organisme, pays, date, lien, CPV) pour toutes les
  annonces ouvertes.
- Après le scoring : champs **riches** (prix, procédure, durée, description, documents) seulement
  pour les annonces jugées **pertinentes**, en enrichissant la requête API.

Remarque : un avis avec une section « Results / Winner » est un **avis d'attribution** (marché déjà
gagné), pas une opportunité. Le filtre `notice-type=cn-standard` les écarte.

## 12. Décisions de conception pour `collecter_ted()`

Pour alimenter la même base que la BCEAO, je produis le même format :

| Champ de ma base | Source TED |
|---|---|
| `cle_unique` | `ted::<publication-number>` |
| `intitule` | `notice-title["eng"]` |
| `source` | `"TED"` |
| organisme / pays | `buyer-name` / `buyer-country` |
| `date_publication` | `publication-date` normalisée |
| `delai_soumission` | `deadline` si présent, normalisé, sinon `null` |
| `lien` | `https://ted.europa.eu/fr/notice/<numéro>` |
| `statut_source` | `"en cours"` (ACTIVE), ou `"clôturé"` si date limite dépassée |

Le prix et les détails riches seront ajoutés **plus tard**, pour les annonces pertinentes.

## 13. Prochaines étapes

1. Écrire `collecter_ted()` avec les champs **essentiels**, testée sur de vraies données.
2. L'intégrer dans `collecte_et_scoring.py`, à côté de `collecter_bceao()`.
3. Vérifier que le scoring repère les vraies annonces cyber.
4. Plus tard : ajouter le prix et les détails pour les pertinentes ; gérer la pagination
   (`iterationNextToken`) si besoin de plus de 100 annonces.